## 工具

#### 加载环境并初始化

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model='deepseek:deepseek-flash'
)

#### [工具调用](https://docs.langchain.com/oss/python/langchain/models#tool-calling)

In [ ]:
from typing import List
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_core.utils.function_calling import convert_to_openai_tool


@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"


# 绑定工具
tools = [get_weather]
tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

# 消息列表
messages: List[BaseMessage] = [
    HumanMessage("今天北京天气如何")
]

# 模型生成工具调用
resp = llm_with_tools.invoke("北京天气如何")
messages.append(resp)

# 工具执行并收集结果
for tool_call in resp.tool_calls:
    tool = tools_by_name[tool_call["name"]]
    tool_result = tool.invoke(tool_call)
    messages.append(tool_result)

# 带工具执行结果再调用模型
final_resp = llm_with_tools.invoke(messages)
print(final_resp.text)

#### 自定义工具属性

In [ ]:
from pprint import pprint
from langchain_core.tools import tool


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """获取指定城市的天气

    Args:
        city: 具体的城市
    """
    return f"{city}天气晴天~"


# 打印工具底层实现的描述
pprint(convert_to_openai_tool(get_weather))

In [ ]:
from pprint import pprint
from langchain_core.tools import tool
from pydantic import BaseModel, Field


class WeatherInput(BaseModel):
    city: str = Field(
        description="具体城市，例如北京",
        min_length=1,
    )
    days: int = Field(
        default=1,
        description="查询天数，范围为1至7天",
        ge=1,
        le=7,
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, days: int = 1) -> str:
    """查询指定城市的天气。"""
    return f"{city}未来{days}天：晴天"


# 打印工具底层实现的描述
pprint(convert_to_openai_tool(get_weather))

#### 扩展参数

##### tool_choice

`none` 意味着模型不会调用任何 tool，而是生成一条消息。

`auto` 意味着模型可以选择生成一条消息或调用一个或多个 tool。

`required` 意味着模型必须调用一个或多个 tool。

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}天气晴天~"


llm_with_tools = llm.bind_tools([get_weather], tool_choice="none")
resp = llm_with_tools.invoke("北京天气如何")

# 不会调用工具
print(resp.model_dump_json(indent=2))